In [2]:
import os
import sys
import tomllib
import pickle
import warnings
import hashlib
import json
from sklearn.exceptions import InconsistentVersionWarning

# Work from the homework directory where all the artifacts live.
# chdir handles relative open() paths; sys.path lets `import model` resolve.
if os.path.basename(os.getcwd()) != 'homework':
    os.chdir('homework')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print(os.getcwd())

from model import normalize_record

# The artifact was pickled with scikit-learn 1.7.2. If the running kernel uses a
# different version, scikit-learn emits an InconsistentVersionWarning on unpickle.
# Silence that specific warning (results were verified against the 1.7.2 reference).
# For an exact reproduction, run this notebook with the pinned env (uv, sklearn 1.7.2).
warnings.simplefilter('ignore', InconsistentVersionWarning)

/Users/marilou/PycharmProjects/ml-zoomcamp-2026/05-Deployment/homework


In [3]:
# UV version
!uv --version

uv 0.11.21 (5aa65dd7a 2026-06-11 aarch64-apple-darwin)


In [4]:
# Scikit-Learn version
with open('pyproject.toml', 'rb') as f:
    deps = tomllib.load(f)['project']['dependencies']
print([d for d in deps if d.startswith('scikit-learn')])

['scikit-learn==1.7.2']


In [5]:
# Conversion probability (Loaded model)
with open('pipeline.bin', 'rb') as f:
    pipeline = pickle.load(f)

lead_q3 = {
    'lead_source': 'paid_ads',
    'industry': 'technology',
    'employment_status': 'employed',
    'location': 'north_america',
    'number_of_courses_viewed': 2,
    'annual_income': 79276.0,
    'interaction_count': 4,
    'lead_score': 0.41,
}

proba_q3 = float(pipeline.predict_proba([normalize_record(lead_q3)])[0, 1])
print(round(proba_q3, 3))

0.533


In [6]:
# Conversion probability (API)

# In terminal to host the server
#!uv run uvicorn predict:app --host 0.0.0.0 --port 9696

!curl -s http://localhost:9696/predict -H 'Content-Type: application/json' -d '{"lead_source": "organic_search", "industry": "technology", "employment_status": "employed", "location": "europe", "number_of_courses_viewed": 4, "annual_income": 80304.0, "interaction_count": 7, "lead_score": 0.74}'

lead_q4 = {
    'lead_source': 'organic_search',
    'industry': 'technology',
    'employment_status': 'employed',
    'location': 'europe',
    'number_of_courses_viewed': 4,
    'annual_income': 80304.0,
    'interaction_count': 7,
    'lead_score': 0.74,
}

proba_q4 = float(pipeline.predict_proba([normalize_record(lead_q4)])[0, 1])
print({'conversion_probability': round(proba_q4, 3), 'conversion': proba_q4 >= 0.5})

{"conversion_probability":0.769799,"conversion":true}{'conversion_probability': 0.77, 'conversion': True}


In [7]:
# Python Base image
with open('Dockerfile') as f:
    first_line = f.readline().strip()
print(first_line)
# Answer Q5: python:3.11.15-slim-bookworm

FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3


In [8]:
# Conversion probability (Docker)
# Run in terminal
# docker build -t zoomcamp-model:2026-hw5 .
# docker run --rm -p 9696:9696 zoomcamp-model:2026-hw5

!python q6_test.py

# Same reference inference as Q4
print('conversion_probability =', round(proba_q4, 3))

# Health check: the served model checksum must match model_metadata.json
model_sha256 = hashlib.sha256(open('pipeline.bin', 'rb').read()).hexdigest()
metadata = json.load(open('model_metadata.json'))
print('checksum matches:', model_sha256 == metadata['model_sha256'])

{'conversion_probability': 0.769799, 'conversion': True}
conversion_probability = 0.77
checksum matches: True
